# 02: 多数据集合并 + 跨数据集质量诊断

将各 per-dataset QC 产出合并为统一的 AnnData，并执行跨数据集 QC 诊断。

**合并策略**：
- 基因交集（inner join）：只保留所有数据集共享的基因，确保跨数据集可比性
- cell_id 全局唯一：per-dataset notebook 已确保 ID 不冲突
- 跨数据集诊断：检查合并后各数据集的 QC 分布是否一致，标记异构性

**输出**：合并后的 `02_merged_v1.h5ad`，作为下游 03-15 的统一入口。

In [ ]:
# === PARAMS ===
PER_DATASET_PATHS = [
    "results/01_nancang_v1.h5ad",
    "results/01_kim_v1.h5ad",
    "results/01_nowicki_v1.h5ad",
    "results/01_yue_v1.h5ad",
]
OUTPUT_PATH = "results/02_merged_v1.h5ad"

JOIN_GENES = "inner"
MIN_SHARED_GENES = 15000
BATCH_KEY = "source_dataset"
DOWNSAMPLE_TO_MIN = False
OUTPUT_VERSION = 1
RANDOM_SEED = 42


In [ ]:
# === Setup ===
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

import scanpy as sc
import anndata
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from pathlib import Path

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')} | anndata {importlib.metadata.version('anndata')}")


## 加载各 per-dataset 产出

In [ ]:
# 加载每个 per-dataset h5ad
adatas = []
dataset_info = []
for fp in PER_DATASET_PATHS:
    if not os.path.exists(fp):
        print(f"WARNING {fp} 不存在，跳过")
        continue
    ad = sc.read_h5ad(fp)
    adatas.append(ad)
    qc_rpt = ad.uns.get("qc_report_v1", {})
    src = ad.obs["source_dataset"].iloc[0] if "source_dataset" in ad.obs.columns else "?"
    dataset_info.append({"路径": fp, "数据集": src, "细胞数": ad.n_obs, "基因数": ad.n_vars,
                         "QC策略": qc_rpt.get("strategy", "?"), "去除细胞": qc_rpt.get("cells_removed", "?"),
                         "去除比例": f"{qc_rpt.get('pct_removed', '?')}%"})
print("===== 各数据集加载摘要 =====")
display(pd.DataFrame(dataset_info))
print(f"\n共加载 {len(adatas)} 个数据集，总细胞数: {sum(a.n_obs for a in adatas):,}")


## 基因空间交集分析

不同数据集的基因空间通常不完全相同。基因交集过小会导致整合分析丢失大量信息。
若交集 < MIN_SHARED_GENES，可能是某个数据集的基因 ID 体系未正确转换。

In [ ]:
# 基因空间交集分析
gene_sets = {}
for ad in adatas:
    src = ad.obs["source_dataset"].iloc[0]
    gene_sets[src] = set(ad.var_names)
all_union = set.union(*gene_sets.values()) if gene_sets else set()
all_inter = all_union.copy()
for gs in gene_sets.values():
    all_inter &= gs
n_union, n_inter = len(all_union), len(all_inter)
print(f"基因并集: {n_union:,}  交集: {n_inter:,} ({100*n_inter/n_union:.1f}%)")
if n_inter < MIN_SHARED_GENES:
    print(f"WARNING: 共享基因 {n_inter:,} < MIN_SHARED_GENES={MIN_SHARED_GENES:,}")
else:
    print(f"OK 共享基因 {n_inter:,} >= {MIN_SHARED_GENES:,}")

# 条形图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
srcs = list(gene_sets.keys()); counts = [len(gene_sets[s]) for s in srcs]
bars = axes[0].bar(range(len(srcs)), counts, color="steelblue", edgecolor="white")
axes[0].set_xticks(range(len(srcs))); axes[0].set_xticklabels(srcs, rotation=30, ha="right")
axes[0].set_ylabel("基因数"); axes[0].set_title("各数据集基因数")
for b, c in zip(bars, counts):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+200, f"{c:,}", ha="center", fontsize=9)
axes[1].bar(["并集","交集"], [n_union, n_inter], color=["lightgray","steelblue"], edgecolor="white", width=0.4)
axes[1].set_ylabel("基因数"); axes[1].set_title(f"基因交集 ({JOIN_GENES} join)")
for i, v in enumerate([n_union, n_inter]):
    axes[1].text(i, v+200, f"{v:,}", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig("results/figures/02_merged_gene_intersection.png", dpi=150, bbox_inches="tight")
plt.show()


## 合并（anndata.concat）

In [ ]:
# 合并所有 per-dataset AnnData
src_keys = [ad.obs["source_dataset"].iloc[0] for ad in adatas]
print(f"合并 {len(adatas)} 个数据集: {src_keys}, join={JOIN_GENES}")
adata = anndata.concat(adatas, join=JOIN_GENES, label="source_dataset", keys=src_keys, index_unique="-")
n_unique = adata.obs_names.nunique()
if n_unique != adata.n_obs:
    print(f"WARNING cell_id 不唯一: {adata.n_obs} obs vs {n_unique} unique")
else:
    print(f"OK cell_id 唯一: {n_unique:,} ids")
print(f"合并后 AnnData: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
display(adata.obs["source_dataset"].value_counts())
for ad in adatas: del ad
adatas.clear(); gc.collect()


## 跨数据集 QC 诊断

绘制三个主 QC 指标按 source_dataset 分组的小提琴图，检查跨数据集的一致性。

**看什么**：
- 各数据集的 n_genes / total_counts / pct_mt 分布范围是否大致可比
- 类器官数据集（Kim, Yue）的 MT% 基线是否确实高于组织活检数据集（Nancang, Nowicki）

**关键原则**：跨数据集 QC 的目标不是"让所有分布一样"，而是"理解并记录差异"。
如果某个数据集的 MT% 基线确实偏高（如类器官固有特性），强行对齐反而是科学错误。

In [ ]:
# 跨数据集 QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    if metric not in adata.obs.columns: continue
    sc.pl.violin(adata, keys=metric, groupby="source_dataset", rotation=30, ax=axes[i], show=False)
    axes[i].set_title(f"{metric}（按 source_dataset）")
plt.tight_layout()
fig.savefig("results/figures/02_merged_qc_by_source.png", dpi=150, bbox_inches="tight")
plt.show()


## 跨数据集摘要表

In [ ]:
# 跨数据集 QC 摘要
print("===== 跨数据集 QC 摘要 =====")
rows = []
for src in sorted(adata.obs["source_dataset"].unique()):
    m = adata.obs["source_dataset"] == src
    sub = adata[m]
    row = {"source_dataset": src, "n_cells": sub.n_obs,
           "median_genes": round(sub.obs["n_genes"].median(), 1),
           "median_umi": round(sub.obs["total_counts"].median(), 1),
           "median_mt_pct": round(sub.obs["pct_counts_mt"].median(), 2)}
    if "predicted_doublet" in sub.obs.columns and sub.obs["predicted_doublet"].notna().any():
        row["doublet_rate_pct"] = round(100 * sub.obs["predicted_doublet"].mean(), 2)
    if "phase" in sub.obs.columns:
        for ph in ["G1","S","G2M"]:
            row[f"phase_{ph}_pct"] = round(100 * (sub.obs["phase"] == ph).mean(), 1)
    rows.append(row)
display(pd.DataFrame(rows))


## QC 异构性记录

In [ ]:
# 记录跨数据集 QC 异构性
qc_reports = {}
qc_strategies = set()
for src in adata.obs["source_dataset"].unique():
    for fp in PER_DATASET_PATHS:
        if not os.path.exists(fp): continue
        tmp = sc.read_h5ad(fp)
        if tmp.obs["source_dataset"].iloc[0] == src:
            rpt = tmp.uns.get("qc_report_v1", {})
            qc_reports[src] = rpt
            qc_strategies.add(rpt.get("strategy", "unknown"))
            del tmp; break
qc_heterogeneous = len(qc_strategies) > 1
print(f"QC 策略: {qc_strategies}, 异构性: {qc_heterogeneous}")
merge_report = {"n_datasets": len(PER_DATASET_PATHS), "join_genes": JOIN_GENES,
    "n_shared_genes": n_inter, "gene_intersection_pct": round(100*n_inter/n_union, 1),
    "qc_heterogeneous": qc_heterogeneous, "qc_strategies": sorted(qc_strategies), "per_dataset_qc": qc_reports}
adata.uns["merge_report_v1"] = merge_report
for k, v in merge_report.items():
    if k != "per_dataset_qc": print(f"  {k}: {v}")
gc.collect()


## 可选：按最小细胞数下采样

In [ ]:
# 可选下采样
if DOWNSAMPLE_TO_MIN:
    min_n = min(adata.obs["source_dataset"].value_counts())
    print(f"下采样到 {min_n} per dataset...")
    np.random.seed(RANDOM_SEED)
    keep_idx = []
    for src in adata.obs["source_dataset"].unique():
        src_idx = adata.obs_names[adata.obs["source_dataset"] == src]
        keep_idx.extend(np.random.choice(src_idx, size=min_n, replace=False))
    adata = adata[keep_idx].copy()
    print(f"下采样后: {adata.n_obs:,} cells")
else:
    print("DOWNSAMPLE_TO_MIN=False")


In [ ]:
# Checkpoint：写入合并 h5ad
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, f"adata.X invariant broken: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
adata.uns["stage"] = "02_merged"
adata.uns["status"] = "experimental"
adata.uns["upstream"] = PER_DATASET_PATHS
adata.uns["version"] = f"v{OUTPUT_VERSION}"
os.makedirs("results", exist_ok=True)
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"OK 写入 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")
del adata; gc.collect()
print("内存已释放。")
